# DeepMTL2R Matryoshka Runner (Kaggle)

Notebook ini menjalankan eksperimen Matryoshka dengan metrik ranking standar dan metrik khusus Effective Dimensionality Efficiency.

Output utama:
- semua metrik per-fold dan agregasi mean±std
- Effective Dimensionality Efficiency per nesting dim
- `matryoshka_summary.json`
- plotting perbandingan metrik dan ringkasan Matryoshka

In [ ]:
!rm -rf /kaggle/working/DeepMTL2R
!git clone https://github.com/jteo0/DeepMTL2R.git

In [ ]:
import os
import sys
import json
import gc
import random
import shutil
import subprocess
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

plt.style.use('seaborn-v0_8-whitegrid')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

In [ ]:
with open(os.devnull, 'w') as fnull:
    with subprocess.Popen([sys.executable, '-m', 'pip', 'install', '-r', '/kaggle/working/DeepMTL2R/requirements.txt'], stdout=fnull, stderr=fnull) as p:
        p.wait()

    with subprocess.Popen([sys.executable, '-m', 'pip', 'install', '-e', '/kaggle/working/DeepMTL2R'], stdout=fnull, stderr=fnull) as p:
        p.wait()

INPUT_DS = '/kaggle/input/mslr-web10k/MSLR-WEB10K'
TARGET_DS = '/kaggle/working/DeepMTL2R/datasets/MSLR-WEB10K'
if os.path.exists(INPUT_DS):
    os.makedirs(TARGET_DS, exist_ok=True)
    shutil.copytree(INPUT_DS, TARGET_DS, dirs_exist_ok=True)

if '/kaggle/working/DeepMTL2R' not in sys.path:
    sys.path.insert(0, '/kaggle/working/DeepMTL2R')

print('Setup complete.')

In [ ]:
PROJECT_ROOT = '/kaggle/working/DeepMTL2R'
EXAMPLE_DIR = os.path.join(PROJECT_ROOT, 'examples', 'MSLR-WEB10K')
DATASET_BASE_PATH = '/kaggle/input/datasets/engkhaledmo/mslr-10k/'

DEBUG_MODE = False
DEBUG_RATIO = 0.0
FOLDS = [1, 2]
TASK_INDICES = [0, 131]
LABEL_INDICES = [131, 132, 133, 135]
REDUCTION_METHOD = 'mean'
USE_MRL = True
USE_GATING = False
MRL_NESTING_DIMS = [32, 64, 128, 256]
EPOCHS_OVERRIDE = 50

OUTPUT_DIR = os.path.join(EXAMPLE_DIR, 'outputs', 'matryoshka')
CHECKPOINT_DIR = os.path.join(EXAMPLE_DIR, 'checkpoints', 'matryoshka')
CONFIG_BASE = os.path.join(EXAMPLE_DIR, 'configs', 'config_matryoshka.json')
CONFIG_OVERRIDE = os.path.join(EXAMPLE_DIR, 'configs', 'config_matryoshka_epochs50.json')
SUMMARY_PATH = os.path.join(OUTPUT_DIR, 'matryoshka_summary.json')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

with open(CONFIG_BASE, 'r', encoding='utf-8') as f:
    _cfg = json.load(f)
_cfg.setdefault('training', {})
_cfg['training']['epochs'] = EPOCHS_OVERRIDE
with open(CONFIG_OVERRIDE, 'w', encoding='utf-8') as f:
    json.dump(_cfg, f, indent=2)

CONFIG_PATH = CONFIG_OVERRIDE

print('PROJECT_ROOT:', PROJECT_ROOT)
print('EXAMPLE_DIR :', EXAMPLE_DIR)
print('DATASET_BASE_PATH:', DATASET_BASE_PATH)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('CHECKPOINT_DIR:', CHECKPOINT_DIR)
print('CONFIG_PATH:', CONFIG_PATH)
print('USE_MRL:', USE_MRL, '| USE_GATING:', USE_GATING)
print('EPOCHS_OVERRIDE:', EPOCHS_OVERRIDE)

In [ ]:
from attr import asdict
from functools import partial

import allrank.models.losses as losses
from allrank.config import Config
from allrank.data.dataset_loading import load_libsvm_dataset, create_data_loaders, PADDED_Y_VALUE
from allrank.models.model import make_model
from allrank.training.train_utils import fit, compute_metrics
from allrank.models.model_utils import get_num_params, CustomDataParallel
from allrank.utils.python_utils import dummy_context_mgr
from allrank.utils.file_utils import create_output_dirs, PathsContainer
from allrank.utils.ltr_logging import init_logger

def summarize_metrics(agg_dict):
    out = {}
    for metric_name, values in agg_dict.items():
        arr = np.array(values, dtype=float)
        out[metric_name] = {
            'mean': float(np.mean(arr)),
            'std': float(np.std(arr, ddof=1)) if len(arr) > 1 else 0.0,
            'per_fold': [float(x) for x in arr.tolist()]
        }
    return out

def evaluate_task_metrics(model, val_dataloader, config, device, task_indices):
    results = {}
    model.eval()
    with torch.no_grad():
        for task_idx in task_indices:
            temp_dl = []
            for xb, yb, indices in val_dataloader:
                all_indices = torch.arange(xb.shape[-1])
                keep_indices = all_indices[~torch.isin(all_indices, torch.tensor(LABEL_INDICES))]
                modified_xb = xb[:, :, keep_indices]
                task_yb = yb if task_idx == 0 else xb[:, :, task_idx]
                task_yb[yb == -1] = -1
                temp_dl.append((modified_xb, task_yb, indices))
            results[task_idx] = compute_metrics(config.metrics, model, temp_dl, device)
    return results

def run_fold(fold):
    dataset_path = os.path.join(DATASET_BASE_PATH, f'Fold{fold}')
    if not os.path.exists(dataset_path):
        raise FileNotFoundError(f'Dataset path not found: {dataset_path}')

    config = Config.from_json(CONFIG_PATH)
    config.model.use_mrl = USE_MRL
    config.model.use_gating = USE_GATING
    config.data.path = dataset_path
    config.loss.args['reduction'] = REDUCTION_METHOD
    config.training.epochs = EPOCHS_OVERRIDE

    max_rows = None
    if DEBUG_MODE and DEBUG_RATIO > 0:
        estimated_total_rows = 30000000
        max_rows = max(1, int(estimated_total_rows * DEBUG_RATIO))

    train_ds, val_ds = load_libsvm_dataset(dataset_path, config.data.slate_length, config.data.validation_ds_role, max_rows=max_rows)
    nf = train_ds.shape[-1] - len(LABEL_INDICES)
    train_dl, val_dl = create_data_loaders(train_ds, val_ds, config.data.num_workers, config.data.batch_size)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = make_model(n_features=nf, **asdict(config.model, recurse=False))
    if torch.cuda.is_available() and torch.cuda.device_count() > 1:
        model = CustomDataParallel(model)
    model.to(device)

    optimizer = getattr(torch.optim, config.optimizer.name)(params=model.parameters(), **config.optimizer.args)
    loss_func = partial(getattr(losses, config.loss.name), **config.loss.args)
    scheduler = getattr(torch.optim.lr_scheduler, config.lr_scheduler.name)(optimizer, **config.lr_scheduler.args) if config.lr_scheduler.name else None

    fold_output_dir = os.path.join(OUTPUT_DIR, f'fold_{fold}')
    os.makedirs(fold_output_dir, exist_ok=True)
    results_filename = os.path.join(fold_output_dir, 'results.txt')

    with (torch.autograd.detect_anomaly() if config.detect_anomaly else dummy_context_mgr()):
        fit_result = fit(
            epochs=config.training.epochs,
            moo_method='ls',
            main_task_index=0,
            task_indices=TASK_INDICES,
            label_indices=LABEL_INDICES,
            results_filename=results_filename,
            model=model,
            loss_func=loss_func,
            task_weights=torch.tensor([1.0 / len(TASK_INDICES)] * len(TASK_INDICES)),
            optimizer=optimizer,
            scheduler=scheduler,
            train_dataloader=train_dl,
            val_dataloader=val_dl,
            config=config,
            gradient_clipping_norm=config.training.gradient_clipping_norm,
            early_stopping_patience=config.training.early_stopping_patience,
            device=device,
            output_dir=fold_output_dir,
            tensorboard_output_path=os.path.join(fold_output_dir, 'tensorboard'),
            use_mrl=USE_MRL,
            use_gating=USE_GATING,
            mrl_nesting_dims=MRL_NESTING_DIMS,
        )

    metrics_by_task = evaluate_task_metrics(model, val_dl, config, device, TASK_INDICES)
    special_metrics = fit_result.get('special_metrics', {})
    num_params = fit_result.get('num_params')

    return {
        'fold': fold,
        'num_params': int(num_params) if num_params is not None else None,
        'metrics_by_task': metrics_by_task,
        'special_metrics': special_metrics,
        'results_filename': results_filename,
        'fold_output_dir': fold_output_dir,
    }

In [ ]:
task_metrics_agg = {task_idx: defaultdict(list) for task_idx in TASK_INDICES}
mrl_rows = []
params_per_fold = []
fold_results = []

for fold in FOLDS:
    print('=' * 80)
    print(f'Processing Fold {fold}')
    print('=' * 80)
    fold_result = run_fold(fold)
    fold_results.append(fold_result)
    params_per_fold.append(fold_result['num_params'])

    for task_idx, task_metrics in fold_result['metrics_by_task'].items():
        for metric_name, metric_value in task_metrics.items():
            task_metrics_agg[task_idx][metric_name].append(float(metric_value))

    mrl_eff = fold_result['special_metrics'].get('mrl_dimensionality_efficiency', {})
    if mrl_eff:
        for task_idx, dim_map in mrl_eff.items():
            for dim, dim_metrics in dim_map.items():
                row = {'fold': fold, 'task_idx': task_idx, 'dim': dim}
                for metric_name, metric_value in dim_metrics.items():
                    if not isinstance(metric_value, dict):
                        row[metric_name] = float(metric_value)
                mrl_rows.append(row)

    del fold_result
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

task_summaries = {task_idx: summarize_metrics(values) for task_idx, values in task_metrics_agg.items()}
mrl_df = pd.DataFrame(mrl_rows)

summary_out = {
    'folds': FOLDS,
    'experiment': 'matryoshka',
    'debug_mode': DEBUG_MODE,
    'epochs_override': EPOCHS_OVERRIDE,
    'params_per_fold': params_per_fold,
    'tasks': task_summaries,
    'mrl_dimensionality_efficiency': mrl_rows,
    'fold_results': fold_results,
}

with open(SUMMARY_PATH, 'w', encoding='utf-8') as f:
    json.dump(summary_out, f, indent=2, default=float)

print('Summary saved to:', SUMMARY_PATH)
for task_idx, summary in task_summaries.items():
    print(f'\nTask {task_idx}:')
    for metric_name in sorted(summary.keys()):
        metric_stats = summary[metric_name]
        print(f"  {metric_name}: {metric_stats['mean']:.4f} ± {metric_stats['std']:.4f}")

if not mrl_df.empty:
    print('\nMatryoshka dimensionality summary:')
    display(mrl_df.head(20))

In [ ]:
for task_idx, summary in task_summaries.items():
    metrics = sorted(summary.keys())
    x = np.arange(len(metrics))
    means = [summary[m]['mean'] for m in metrics]
    stds = [summary[m]['std'] for m in metrics]
    plt.figure(figsize=(max(10, len(metrics) * 1.1), 4.5))
    plt.bar(x, means, yerr=stds, capsize=4)
    plt.xticks(x, metrics, rotation=45, ha='right')
    plt.ylabel('Score')
    plt.title(f'Matryoshka task {task_idx} metrics (mean ± std)')
    plt.tight_layout()
    plt.show()

if not mrl_df.empty:
    metric_cols = [c for c in mrl_df.columns if c not in {'fold', 'task_idx', 'dim'}]
    if 'ndcg_30' in metric_cols:
        pivot = mrl_df.pivot_table(index='dim', values='ndcg_30', aggfunc='mean').sort_index()
        plt.figure(figsize=(8, 4))
        plt.plot(pivot.index.astype(str), pivot['ndcg_30'], marker='o')
        plt.xlabel('Nesting dim')
        plt.ylabel('NDCG@30')
        plt.title('Matryoshka Effective Dimensionality Efficiency')
        plt.tight_layout()
        plt.show()

In [ ]:
ARCHIVE_DIR = os.path.join(PROJECT_ROOT, 'archives')
os.makedirs(ARCHIVE_DIR, exist_ok=True)
checkpoint_zip = shutil.make_archive(os.path.join(ARCHIVE_DIR, 'matryoshka_checkpoints'), 'zip', CHECKPOINT_DIR)
results_zip = shutil.make_archive(os.path.join(ARCHIVE_DIR, 'matryoshka_results'), 'zip', OUTPUT_DIR)
print('Checkpoint ZIP:', checkpoint_zip)
print('Results ZIP:', results_zip)

## Catatan
- Notebook ini khusus Matryoshka.
- Metrik khusus yang ditampilkan adalah Effective Dimensionality Efficiency.
- Jika ingin debug cepat, ubah `DEBUG_MODE` dan `EPOCHS_OVERRIDE` di cell konfigurasi.